# Visualización de Resultados LOO Blind — TT_Ratones_2026

Notebook para generar las figuras de validación Leave-One-Out (LOO) blind para la tesis.

**Fuente de datos:**
- `loo_resultados.csv` — F1 por video (13 videos) × método (SimBA, B-SOiD, Ensemble OR) — extraído de `reportes/01_ESTADO_ACTUAL.md` §5.
- `loo_detalle_simba.csv` — Precisión/Recall/F1 + ground truth por video (5 videos con log disponible).
- `checkpoint_M1_conditional.csv` — Ensemble condicional sobre 5 videos críticos.

**Outputs:** PNG (300 dpi) y SVG en `reportes/figuras/`. Listos para insertar en Word o LaTeX.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path

HERE = Path.cwd()
if HERE.name != 'figuras':
    candidate = HERE / 'reportes' / 'figuras'
    if candidate.exists():
        HERE = candidate
print('Working dir:', HERE)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)
mpl.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans',
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLOR_SIMBA = '#1f77b4'
COLOR_BSOID = '#d62728'
COLOR_ENSEMBLE = '#2ca02c'
COLOR_CONDITIONAL = '#9467bd'
PALETA_METODOS = {
    'SimBA RF': COLOR_SIMBA,
    'B-SOiD': COLOR_BSOID,
    'Ensemble OR': COLOR_ENSEMBLE,
    'Conditional': COLOR_CONDITIONAL,
}

def save(fig, name):
    out_png = HERE / f'{name}.png'
    out_svg = HERE / f'{name}.svg'
    fig.savefig(out_png)
    fig.savefig(out_svg)
    print(f'  saved {out_png.name} + {out_svg.name}')

df = pd.read_csv(HERE / 'loo_resultados.csv')
df_det = pd.read_csv(HERE / 'loo_detalle_simba.csv')
df_m1 = pd.read_csv(HERE / 'checkpoint_M1_conditional.csv')
df

## Figura 1 — Barras agrupadas: F1 Grooming por video y método
Compara los tres clasificadores en cada uno de los 13 videos LOO blind. Resalta la varianza entre animales y los rescates de B-SOiD donde SimBA colapsa.

In [ ]:
df_long_g = df.melt(
    id_vars='video',
    value_vars=['grooming_simba_f1', 'grooming_bsoid_f1', 'grooming_ensemble_or_f1'],
    var_name='metodo', value_name='F1'
)
df_long_g['metodo'] = df_long_g['metodo'].map({
    'grooming_simba_f1': 'SimBA RF',
    'grooming_bsoid_f1': 'B-SOiD',
    'grooming_ensemble_or_f1': 'Ensemble OR',
})

fig, ax = plt.subplots(figsize=(11, 4.8))
sns.barplot(data=df_long_g, x='video', y='F1', hue='metodo', palette=PALETA_METODOS, ax=ax, edgecolor='white')
ax.axhline(0.85, color='gray', linestyle='--', linewidth=1, label='Umbral deployable (0.85)')
ax.set_title('F1-score Grooming por video — Validación LOO blind (13 videos)')
ax.set_xlabel('Video (animal held-out)')
ax.set_ylabel('F1-score blind')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=35)
for tl in ax.get_xticklabels():
    tl.set_horizontalalignment('right')
ax.legend(loc='upper right', frameon=True, fontsize=9)
save(fig, 'fig1_f1_grooming_barras')
plt.show()

## Figura 2 — Distribución (boxplot + strip) de F1 por método
Visualiza la varianza global del F1 entre videos. Demuestra que el ensemble reduce la cola izquierda (colapsos a F1≈0).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.6))
orden = ['SimBA RF', 'B-SOiD', 'Ensemble OR']
sns.boxplot(data=df_long_g, x='metodo', y='F1', order=orden, hue='metodo',
            palette=PALETA_METODOS, ax=ax, width=0.55, fliersize=0,
            linewidth=1.2, legend=False)
sns.stripplot(data=df_long_g, x='metodo', y='F1', order=orden,
              color='black', alpha=0.55, size=5, jitter=0.12, ax=ax)

means = df_long_g.groupby('metodo')['F1'].mean()
for i, met in enumerate(orden):
    ax.annotate(f'μ={means[met]:.2f}', (i, means[met]), xytext=(12, 0),
                textcoords='offset points', va='center', fontsize=10, fontweight='bold')
ax.set_title('Distribución F1 Grooming por método (n=13 videos)')
ax.set_xlabel('')
ax.set_ylabel('F1-score blind')
ax.set_ylim(-0.05, 1.05)
save(fig, 'fig2_distribucion_f1_metodos')
plt.show()

## Figura 3 — Comparación Grooming vs Thigmotaxis (SimBA solo)
Yuxtapone ambas conductas en los mismos 13 videos para evidenciar que Thigmotaxis es más estable y Grooming sufre del problema de muestra pequeña.

In [ ]:
df_comp = df[['video', 'grooming_simba_f1', 'thigmotaxis_simba_f1']].copy()
df_comp_long = df_comp.melt(id_vars='video', var_name='conducta', value_name='F1')
df_comp_long['conducta'] = df_comp_long['conducta'].map({
    'grooming_simba_f1': 'Grooming',
    'thigmotaxis_simba_f1': 'Thigmotaxis',
})
fig, ax = plt.subplots(figsize=(11, 4.4))
sns.barplot(data=df_comp_long, x='video', y='F1', hue='conducta',
            palette={'Grooming': '#e377c2', 'Thigmotaxis': '#17becf'}, ax=ax, edgecolor='white')
for cond, c in [('Grooming', '#e377c2'), ('Thigmotaxis', '#17becf')]:
    m = df_comp_long[df_comp_long['conducta'] == cond]['F1'].mean()
    ax.axhline(m, color=c, linestyle=':', linewidth=1.4, alpha=0.8, label=f'Promedio {cond}: {m:.2f}')
ax.set_title('F1 SimBA — Grooming vs Thigmotaxis por video (LOO blind)')
ax.set_xlabel('Video (animal held-out)')
ax.set_ylabel('F1-score blind')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=35)
for tl in ax.get_xticklabels():
    tl.set_horizontalalignment('right')
ax.legend(loc='upper right', frameon=True, fontsize=9)
save(fig, 'fig3_grooming_vs_thigmotaxis')
plt.show()

## Figura 4 — Precisión vs Recall (SimBA blind)
Para los 5 videos donde tenemos logs detallados con P/R por separado. Muestra el trade-off del clasificador: algunos videos tienen precisión perfecta pero recall bajo (modelo *tímido*), otros lo contrario.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.0), sharex=True, sharey=True)

g = df_det.dropna(subset=['grooming_precision', 'grooming_recall'])
axes[0].scatter(g['grooming_recall'], g['grooming_precision'], s=110, c=COLOR_SIMBA,
                edgecolors='black', linewidth=0.6, zorder=3)
for _, r in g.iterrows():
    axes[0].annotate(r['video'], (r['grooming_recall'], r['grooming_precision']),
                     xytext=(8, 5), textcoords='offset points', fontsize=9)
axes[0].set_title('Grooming')
axes[0].set_xlabel('Recall (exhaustividad)')
axes[0].set_ylabel('Precisión')

t = df_det.dropna(subset=['thigmotaxis_precision', 'thigmotaxis_recall'])
axes[1].scatter(t['thigmotaxis_recall'], t['thigmotaxis_precision'], s=110, c='#17becf',
                edgecolors='black', linewidth=0.6, zorder=3)
for _, r in t.iterrows():
    axes[1].annotate(r['video'], (r['thigmotaxis_recall'], r['thigmotaxis_precision']),
                     xytext=(8, 5), textcoords='offset points', fontsize=9)
axes[1].set_title('Thigmotaxis')
axes[1].set_xlabel('Recall (exhaustividad)')

for ax in axes:
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    for f1_iso in [0.2, 0.4, 0.6, 0.8]:
        x = np.linspace(0.01, 1.0, 200)
        y = (f1_iso * x) / (2 * x - f1_iso)
        m = (y > 0) & (y <= 1)
        ax.plot(x[m], y[m], color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
        ax.text(1.0, f1_iso / (2 - f1_iso), f'F1={f1_iso}', color='gray', fontsize=8, va='bottom', ha='right')

fig.suptitle('Precisión vs Recall en LOO blind (SimBA RF, n=5 videos con log disponible)', fontweight='bold')
fig.tight_layout()
save(fig, 'fig4_precision_vs_recall')
plt.show()

## Figura 5 — Desbalance de clases por video
Visualiza el porcentaje de frames con Grooming y Thigmotaxis por video. Sustenta argumento de la "trampa de exactitud".

In [ ]:
df_pct = df_det[['video', 'grooming_gt_pct', 'thigmotaxis_gt_pct']].copy()
df_pct_long = df_pct.melt(id_vars='video', var_name='conducta', value_name='pct')
df_pct_long['conducta'] = df_pct_long['conducta'].map({
    'grooming_gt_pct': 'Grooming',
    'thigmotaxis_gt_pct': 'Thigmotaxis',
})
fig, ax = plt.subplots(figsize=(8.5, 4.3))
sns.barplot(data=df_pct_long, x='video', y='pct', hue='conducta',
            palette={'Grooming': '#e377c2', 'Thigmotaxis': '#17becf'}, ax=ax, edgecolor='white')
ax.axhline(8.5, color='#e377c2', linestyle='--', linewidth=1.0, alpha=0.7, label='Grooming promedio dataset (8.5%)')
ax.axhline(3.1, color='#17becf', linestyle='--', linewidth=1.0, alpha=0.7, label='Thigmotaxis promedio dataset (3.1%)')
ax.set_title('Desbalance de clases — % de frames con conducta positiva')
ax.set_xlabel('Video')
ax.set_ylabel('% frames con conducta')
ax.tick_params(axis='x', rotation=20)
ax.legend(loc='upper right', frameon=True, fontsize=8)
save(fig, 'fig5_desbalance_clases')
plt.show()

## Figura 6 — La Trampa de la Exactitud
Demuestra visualmente por qué Accuracy es engañosa: un clasificador trivial (predice siempre 0) obtiene 91.5% accuracy en Grooming pero F1=0.

In [ ]:
TOTAL = 243253
GROOM_POS = 20757
GROOM_NEG = TOTAL - GROOM_POS

modelos = ['Modelo trivial\n(predice 0)', 'SimBA RF\nLOO blind', 'Ensemble OR\nLOO blind']
accuracy_groom = [GROOM_NEG / TOTAL, 0.91, 0.92]
f1_groom = [0.0, 0.45, 0.56]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
x = np.arange(len(modelos))
w = 0.36

axes[0].bar(x - w/2, accuracy_groom, w, color='#bcbd22', label='Accuracy', edgecolor='white')
axes[0].bar(x + w/2, f1_groom, w, color=COLOR_SIMBA, label='F1-score', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(modelos, fontsize=9)
axes[0].set_ylabel('Métrica')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Grooming: Accuracy vs F1')
axes[0].legend(loc='upper left', frameon=True)
for i, (a, f) in enumerate(zip(accuracy_groom, f1_groom)):
    axes[0].text(i - w/2, a + 0.015, f'{a:.2f}', ha='center', fontsize=9)
    axes[0].text(i + w/2, f + 0.015, f'{f:.2f}', ha='center', fontsize=9)

labels_pie = ['Grooming positivo\n(8.5%)', 'Reposo / no-Grooming\n(91.5%)']
sizes = [GROOM_POS, GROOM_NEG]
colors_pie = ['#e377c2', '#dddddd']
axes[1].pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title(f'Composición del dataset Grooming\n(n={TOTAL:,} frames, 26 videos)', fontsize=11)

fig.suptitle('La "Trampa de la Exactitud" — por qué F1 es la métrica correcta', fontweight='bold', y=1.02)
fig.tight_layout()
save(fig, 'fig6_trampa_exactitud')
plt.show()

## Figura 7 — Ensemble condicional vs alternativas (5 videos críticos)
Datos del checkpoint M1: muestra el efecto del rescate condicional con B-SOiD donde SimBA colapsa.

In [ ]:
df_m1_long = df_m1.melt(
    id_vars='video',
    value_vars=['simba_f1', 'bsoid_f1', 'ensemble_or_f1', 'conditional_f1_min100'],
    var_name='metodo', value_name='F1'
)
df_m1_long['metodo'] = df_m1_long['metodo'].map({
    'simba_f1': 'SimBA RF',
    'bsoid_f1': 'B-SOiD',
    'ensemble_or_f1': 'Ensemble OR',
    'conditional_f1_min100': 'Conditional',
})
df_m1_long['video_short'] = df_m1_long['video'].str.split('_').str[0]

fig, ax = plt.subplots(figsize=(10, 4.6))
sns.barplot(data=df_m1_long, x='video_short', y='F1', hue='metodo',
            palette=PALETA_METODOS, ax=ax, edgecolor='white')
ax.set_title('Ensemble condicional vs alternativas — 5 videos críticos (M1)')
ax.set_xlabel('Video')
ax.set_ylabel('F1-score Grooming blind')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', frameon=True, fontsize=9, ncol=2)
save(fig, 'fig7_ensemble_condicional_M1')
plt.show()

## Tabla resumen para tesis
Estadísticas agregadas por método.

In [ ]:
resumen = df_long_g.groupby('metodo')['F1'].agg(['mean', 'std', 'min', 'max', 'count']).round(3)
resumen.columns = ['Promedio', 'Desv.Est', 'Mínimo', 'Máximo', 'n videos']
resumen = resumen.reindex(['SimBA RF', 'B-SOiD', 'Ensemble OR'])

resumen_thig = df[['thigmotaxis_simba_f1']].agg(['mean', 'std', 'min', 'max', 'count']).T.round(3)
resumen_thig.index = ['Thigmotaxis SimBA RF']
resumen_thig.columns = resumen.columns

tabla = pd.concat([resumen, resumen_thig])
tabla.to_csv(HERE / 'tabla_resumen_loo.csv')
print('Tabla resumen guardada en tabla_resumen_loo.csv')
tabla

---
## Exportación para tesis

PNG (300 dpi) ya son listos para Word. SVG para LaTeX. Si necesitas TIFF para revistas: agregar `fig.savefig('...tiff', dpi=600)` en `save()`.

**Archivos generados en `reportes/figuras/`:**
- `fig1_f1_grooming_barras.{png,svg}`
- `fig2_distribucion_f1_metodos.{png,svg}`
- `fig3_grooming_vs_thigmotaxis.{png,svg}`
- `fig4_precision_vs_recall.{png,svg}`
- `fig5_desbalance_clases.{png,svg}`
- `fig6_trampa_exactitud.{png,svg}`
- `fig7_ensemble_condicional_M1.{png,svg}`
- `tabla_resumen_loo.csv`